# 02 — Caching and State

Soma uses **content-addressable caching** — a SHA-256 hash of the computation
inputs determines the cache key. This means:

- Same config + same data → same cache key → cached result reused
- Change a parameter → different hash → recomputation

This notebook explores:
- How config hashing works
- The three types of data on a Filter (parameters, state, internal)
- How state flows through a pipeline
- Cache invalidation by cascade

## 2.1 — The three types of data

Every filter has a clear separation:

```
Filter
├── Parameters (public attrs)     → config_hash → cache key
│   e.g. self.degree = 3
├── State (returned by fit())     → passed to forward() as argument
│   e.g. {"mean": 20.0, "std": 14.14}
└── Internal (private _ attrs)    → invisible to cache system
    e.g. self._call_count
```

This separation is what makes Soma's caching work:
- **State cache key** = `hash(config_hash + training_data_hash)`
- **Output cache key** = `hash(config_hash + state_hash + input_hash)`

In [ ]:
from soma import Filter, Pipeline

class Scaler(Filter):
    """Demonstrates the three types of data."""

    def __init__(self, factor=1.0):
        super().__init__(factor=factor)  # parameter: affects config hash
        self._call_count = 0             # internal: invisible to cache

    def fit(self, x, y=None):
        self._call_count += 1
        mean = sum(x) / len(x)
        return {"mean": mean}            # state: learned, passed to forward

    def forward(self, x, state):
        self._call_count += 1
        return [(v - state["mean"]) * self.factor for v in x]

# Two scalers with different parameters → different cache keys
s1 = Scaler(factor=1.0)
s2 = Scaler(factor=2.0)

# Same training data, different parameters
state1 = s1.fit([10.0, 20.0, 30.0])
state2 = s2.fit([10.0, 20.0, 30.0])

print(f"Same state (same data):      {state1} == {state2}: {state1 == state2}")
print(f"Different factor → different outputs:")
print(f"  factor=1.0: {s1.forward([15.0], state1)}")
print(f"  factor=2.0: {s2.forward([15.0], state2)}")
print(f"Internal _call_count: {s1._call_count} (not part of cache key)")

## 2.2 — How config hashing works

The config hash is computed from the **class name** + **sorted public attributes** (excluding `_` prefix).
This is what determines cache identity for a filter.

In [ ]:
import json

def show_config_hash_inputs(f):
    """Show what goes into a filter's config hash."""
    class_name = type(f).__name__
    public_attrs = {k: v for k, v in f.__dict__.items() if not k.startswith("_")}
    sorted_json = json.dumps(public_attrs, sort_keys=True)
    print(f"  Class:      {class_name}")
    print(f"  Public:     {public_attrs}")
    print(f"  Hash input: {class_name} + {sorted_json}")

# Same class, same params → same config hash
a = Scaler(factor=2.0)
b = Scaler(factor=2.0)
print("Scaler(factor=2.0):")
show_config_hash_inputs(a)
print()

# Different params → different config hash
c = Scaler(factor=3.0)
print("Scaler(factor=3.0):")
show_config_hash_inputs(c)
print()

# Internal attrs are excluded
a._call_count = 999  # does NOT affect config hash
print("After changing _call_count:")
show_config_hash_inputs(a)

## 2.3 — State flows through the pipeline

When a pipeline runs `fit()`, state flows forward — each filter's output
becomes the next filter's input. The pipeline stores each filter's state
internally for later use during `predict()`.

```
fit(x):    x → [fit₁ + fwd₁] → x₂ → [fit₂ + fwd₂] → x₃
predict(x): x → [fwd₁(state₁)] → x₂ → [fwd₂(state₂)] → x₃
```

In [ ]:
class TracingFilter(Filter):
    """Logs what it receives to help visualize data flow."""

    def __init__(self, name="?"):
        super().__init__(name=name)

    def fit(self, x, y=None):
        print(f"  [{self.name}] fit() received: {x}")
        mean = sum(x) / len(x) if isinstance(x, list) else 0
        return {"mean": mean}

    def forward(self, x, state):
        print(f"  [{self.name}] forward() received: x={x}, state={state}")
        result = [v - state["mean"] for v in x]
        print(f"  [{self.name}] forward() returns:  {result}")
        return result

print("=== fit() phase ===")
pipe = Pipeline([
    ("center_1", TracingFilter(name="center_1")),
    ("center_2", TracingFilter(name="center_2")),
])
pipe.fit([10.0, 20.0, 30.0])

print("\n=== predict() phase ===")
pipe.predict([100.0, 200.0])

## 2.4 — Cascade invalidation

In a pipeline A → B → C, if you change A's parameters:
- A's config hash changes → A's state is recomputed
- A's new output becomes B's new input → B's state is invalidated
- B's new output cascades to C → C is also invalidated

This is automatic — you never need to manually invalidate caches.

In [ ]:
class Multiplier(Filter):
    """Multiplies by a configurable factor."""
    def __init__(self, factor=2.0):
        super().__init__(factor=factor)

    def fit(self, x, y=None):
        return {}

    def forward(self, x, state):
        return [v * self.factor for v in x]

class Summer(Filter):
    """Adds a learned offset (mean of training data)."""
    def fit(self, x, y=None):
        return {"offset": sum(x) / len(x)}

    def forward(self, x, state):
        return [v + state["offset"] for v in x]

# Pipeline: multiply → sum with offset
data = [1.0, 2.0, 3.0]

# Run 1: factor=2
pipe_v1 = Pipeline([Multiplier(factor=2.0), Summer()])
pipe_v1.fit(data)
result_v1 = pipe_v1.predict([10.0])
print(f"factor=2.0 → predict(10): {result_v1}")
# 10*2=20, offset=mean([2,4,6])=4, 20+4=24

# Run 2: change factor to 3 — cascading invalidation
# Multiplier's output changes → Summer gets different training data → different offset
pipe_v2 = Pipeline([Multiplier(factor=3.0), Summer()])
pipe_v2.fit(data)
result_v2 = pipe_v2.predict([10.0])
print(f"factor=3.0 → predict(10): {result_v2}")
# 10*3=30, offset=mean([3,6,9])=6, 30+6=36

print(f"\nChanging Multiplier.factor cascaded through to Summer's learned state.")

## 2.5 — Summary: Cache key hierarchy

```
CacheKey::for_state(config_hash, data_hash)
    ↓
    state_hash = hash(state)
    ↓
CacheKey::for_output(config_hash, state_hash, input_hash)
```

| What changes | What gets invalidated |
|---|---|
| Filter parameter | State + all downstream outputs |
| Training data | State + all downstream outputs |
| Input data (predict) | Only that filter's output |
| Internal `_` attribute | Nothing (invisible to cache) |

---

**Next:** [03 — Search and Optimization](./03_search_and_optimization.ipynb) — define search spaces and run hyperparameter studies.